In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np
import pandas as pd
import pingouin as pg
from natsort import natsorted
import plotly.graph_objects as go
from os.path import join as pjoin

sys.path.append("../..")
import circletrack_behavior as ctb
import plotting_functions as pf

def set_groups(df, grouping=1): 
    df['group'] = df['mouse']
    if grouping == 1:
        df['group'] = df['group'].replace({'Z38': 'WT', 'Z40': 'WT', 'Z33': 'WT', 'Z34': 'APP-KI', 'Z35': 'APP-KI',
                                        'Z36': 'WT', 'Z37': 'APP-KI', 'Z46': 'APP-KI', 'Z47': 'APP-KI', 'Z48': 'WT',
                                        'Z49': 'WT', 'Z51': 'APP-KI', 'Z52': 'WT', 'Z53': 'APP-KI', 'Z54': 'APP-KI',
                                        'Z63': 'WT', 'Z64': 'APP-KI', 'Z66': 'WT'})
    elif grouping == 2:
        df['group'] = df['group'].replace({'Z38': 'Wildtype Male', 'Z40': 'Wildtype Male', 'Z33': 'Wildtype Male', 'Z34': 'APP-KI Male', 'Z35': 'APP-KI Female',
                                            'Z36': 'Wildtype Female', 'Z37': 'APP-KI Female', 'Z46': 'APP-KI Male', 'Z47': 'APP-KI Female', 'Z48': 'Wildtype Female',
                                            'Z49': 'Wildtype Female', 'Z51': 'APP-KI Male', 'Z52': 'Wildtype Male', 'Z53': 'APP-KI Male', 'Z54': 'APP-KI Male',
                                            'Z63': 'Wildtype Male', 'Z64': 'APP-KI Male', 'Z66': 'Wildtype Male'})
    return df

In [ ]:
## Set path variables
behavior_type = 'circletrack_data'
csv_name = 'circle_track.csv'
dpath = os.path.abspath(f'../../../AD_Updating/AD_Updating1/{behavior_type}/data/**/**/**/{csv_name}')
fig_path = ('../../../AD_Updating/AD_Updating1/intermediate_figures/')
group_colors = ['red', 'midnightblue']
four_group_colors = ['orchid', 'blue', 'darkorchid', 'darkblue']
group_colors_dict = {'WT': 'midnightblue', 'APP-KI': 'red'}
chance_color = 'darkgrey'
## Create file list dataframe
file_list = ctb.get_file_list(dpath)
mouse_id = []
for file in file_list:
    mouse_id.append(ctb.get_mouse(file, str2match='(Z[0-9]+)'))
combined = ctb.combine(file_list, mouse_id)

In [ ]:
lick_df = pd.DataFrame()
for mouse in np.unique(mouse_id):
    lick_data = ctb.get_lick_accuracy(combined, mouse)
    lick_df = pd.concat([lick_df, lick_data], ignore_index=True)
lick_df = set_groups(lick_df)
lick_df = lick_df[(lick_df['mouse'] != 'Z40') & (lick_df['mouse'] != 'Z34')]
## Lick accuracy plot
fig = pf.plot_behavior_across_days(lick_df, x_var='day', y_var='percent_correct', groupby_var=['day', 'group'], plot_transitions=[12.5, 24.5],
                                   marker_color=group_colors, avg_color='darkgrey', plot_datapoints=False, transition_color=[chance_color, chance_color],
                                   x_title='Day', y_title='Lick Accuracy (%)', titles=['Circle Track: Lick Accuracy'], height=800, width=800)
fig.show()
fig.write_image(pjoin(fig_path, 'lick_accuracy.png'))

In [ ]:
## Repeated measures ANOVA during reversal
anova_data = lick_df[(lick_df['day'] >= 13) & (lick_df['day']) < 25]
anova_data.mixed_anova(dv='percent_correct', within='day', between='group', subject='mouse')

In [ ]:
reward_df = pd.DataFrame()
for mouse in np.unique(mouse_id):
    reward_data = ctb.get_total_rewards(combined, mouse)
    reward_df = pd.concat([reward_df, reward_data], ignore_index=True)
reward_df = set_groups(reward_df)
reward_df = reward_df[(reward_df['mouse'] != 'Z40') & (reward_df['mouse'] != 'Z34')]
## Total rewards plot
fig = pf.plot_behavior_across_days(reward_df, x_var='day', y_var='total_rewards', groupby_var=['day', 'group'], plot_transitions=[12.5, 24.5],
                                   marker_color=group_colors, avg_color='darkgrey', expert_line=False, chance=False, plot_datapoints=False,
                                   transition_color=[chance_color, chance_color],
                                   x_title='Day', y_title='Total Rewards', titles=['Circle Track: Rewards'], height=800, width=800)
fig.show()
fig.write_image(pjoin(fig_path, 'total_rewards.png'))

In [ ]:
## Repeated measures ANOVA during reversal
anova_data = reward_df[reward_df['day'] >= 1]
anova_data.mixed_anova(dv='total_rewards', within='day', between='group', subject='mouse')

In [ ]:
lick_df = pd.DataFrame()
for mouse in np.unique(mouse_id):
    lick_data = ctb.get_lick_accuracy(combined, mouse)
    lick_df = pd.concat([lick_df, lick_data], ignore_index=True)
lick_df = set_groups(lick_df, grouping=2)
## Lick accuracy plot
fig = pf.plot_behavior_across_days(lick_df, x_var='day', y_var='percent_correct', groupby_var=['day', 'group'], plot_transitions=[12.5],
                                   marker_color=four_group_colors, avg_color='darkgrey', plot_datapoints=False,
                                   x_title='Day', y_title='Lick Accuracy (%)', titles=['Circle Track: Lick Accuracy'], height=800, width=800)
fig.show()
fig.write_image(pjoin(fig_path, 'lick_accuracy_mf.png')) 

In [ ]:
reward_df = pd.DataFrame()
for mouse in np.unique(mouse_id):
    reward_data = ctb.get_total_rewards(combined, mouse)
    reward_df = pd.concat([reward_df, reward_data], ignore_index=True)
reward_df = set_groups(reward_df, grouping=2)
## Total rewards plot
fig = pf.plot_behavior_across_days(reward_df, x_var='day', y_var='total_rewards', groupby_var=['day', 'group'], plot_transitions=[12.5],
                                   marker_color=four_group_colors, avg_color='darkgrey', expert_line=False, chance=False, plot_datapoints=False,
                                   x_title='Day', y_title='Total Rewards', titles=['Circle Track: Rewards'], height=800, width=800)
fig.show()
fig.write_image(pjoin(fig_path, 'total_rewards_mf.png'))

### Load settings

In [ ]:
## Settings
lin_path = '../../../AD_Updating/AD_Updating1/output/lin_behav/'
circle_path = '../../../AD_Updating/AD_Updating1/output/behav/'
fig_path = '../../../AD_Updating/AD_Updating1/intermediate_figures'
wt_mice = ['Z38', 'Z40', 'Z33', 'Z36', 'Z48', 'Z49', 'Z52', 'Z63', 'Z66']
app_mice = ['Z34', 'Z35', 'Z37', 'Z46', 'Z47', 'Z51', 'Z53', 'Z54', 'Z64']
male_mice = ['Z38', 'Z40', 'Z33', 'Z34', 'Z46', 'Z51', 'Z52', 'Z53', 'Z54', 'Z63', 'Z64', 'Z66']
two_color_plots = ['red', 'midnightblue']
group_colors_dict = {'WT': 'midnightblue', 'APP-KI': 'red', 'Wildtype Male': 'darkblue', 'APP-KI Male': 'blue',
                     'Wildtype Female': 'darkorchid', 'APP-KI Female': 'orchid', 'Wildtype': 'midnightblue'}
chance_color = 'darkgrey'
avg_color = 'midnightblue'
subject_color = ['red', 'midnightblue']
symbols_list = ['circle', 'square']
excluded_mice = ['Z34', 'Z40']

### Linear track data

In [ ]:
result_dict = {'mouse': [], 'day': [], 'group': [], 'rewards': []}
for mouse in os.listdir(lin_path):
    mouse_path = pjoin(lin_path, mouse)
    group = 'WT' if mouse in wt_mice else 'APP-KI'
    for idx, session in enumerate(os.listdir(mouse_path)):
        behav = pd.read_feather(pjoin(mouse_path, f'{session}'))
        result_dict['mouse'].append(mouse)
        result_dict['day'].append(idx+1)
        result_dict['group'].append(group)
        result_dict['rewards'].append(np.sum(behav['water']))
results_df = pd.DataFrame(result_dict)

In [ ]:
## Plot rewards across days between wt and app
fig = pf.plot_behavior_across_days(results_df, x_var='day', y_var='rewards', groupby_var=['day', 'group'], plot_transitions=None,
                                   marker_color=two_color_plots, avg_color='darkgrey', expert_line=False, chance=False,
                                   x_title='Day', y_title='Rewards', titles=['Linear Track'], height=500, width=500)
fig.show()
results_df.mixed_anova(dv='rewards', within='day', subject='mouse', between='group')
fig.write_image(pjoin(fig_path, 'linear_track_rewards.png'))

### Lick accuracy and rewards earned.

In [ ]:
## Circle track behavior
lick_thresh = 3
data_of_interest = 'behav' ## one of behav, aligned_minian, lin_behav
circ_dict = {'mouse': [], 'experiment': [], 'sex': [], 'group': [], 'day': [], 'session': [], 'rewards': [], 'percent_correct': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass 
    else:
        mpath = pjoin(circle_path, mouse)
        sex = 'Male' if mouse in male_mice else 'Female'
        group = 'WT' if mouse in wt_mice else 'APP-KI'
        for idx, session in enumerate(natsorted(os.listdir(mpath))):
            behav = pd.read_feather(pjoin(mpath, session))
            behav = behav[~behav['probe']] ## exclude probe
            reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]    
            pc = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=lick_thresh, by_trials=False)
            circ_dict['mouse'].append(mouse)
            circ_dict['experiment'].append(behav['cohort'].unique()[0])
            circ_dict['sex'].append(sex)
            circ_dict['group'].append(group)
            circ_dict['day'].append(idx+1)
            circ_dict['session'].append(behav['session'].unique()[0])
            circ_dict['rewards'].append(np.sum(behav['water']))
            circ_dict['percent_correct'].append(pc)
ct_df = pd.DataFrame(circ_dict)

In [ ]:
## Plot lick accuracy 
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='percent_correct', groupby_var=['day', 'group'], plot_transitions=[12.5, 24.5], symbols=symbols_list,
                                   marker_color=subject_color, avg_color=avg_color, expert_line=False, chance=True, transition_color=['darkgrey', 'darkgrey'],
                                   plot_datapoints=False, x_title='Day', y_title='Lick Accuracy (%)', titles=[], height=500, width=500)
fig.update_yaxes(range=[0, 101])
fig.show()

### Analyze probe information.

In [ ]:
lick_dict_probe = {'mouse': [], 'day': [], 'group': [], 'sex': [], 'session': [], 'num_licks': [], 'probe_pc': [], 'session_pc': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass
    else:
        mouse_path = pjoin(circle_path, mouse)
        group = 'WT' if mouse in wt_mice else 'APP-KI'
        sex = 'Male' if mouse in male_mice else 'Female'
        for idx, session in enumerate(os.listdir(mouse_path)):
            behav = pd.read_feather(pjoin(mouse_path, f'{session}'))
            if (behav['session'].to_numpy()[0] == 'AP') | (behav['session'].to_numpy()[0] == 'RP'):
                behav_probe = behav.loc[behav['probe']]
                behav_no_probe = behav[~behav['probe']]
                reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
                ## Percent correct licking
                pc = ctb.lick_accuracy(behav_probe, port_one=reward_one, port_two=reward_two, by_trials=False)
                session_pc = ctb.lick_accuracy(behav_no_probe, port_one=reward_one, port_two=reward_two, by_trials=False)
                lick_dict_probe['mouse'].append(mouse)
                lick_dict_probe['day'].append(idx+1)
                lick_dict_probe['group'].append(group)
                lick_dict_probe['sex'].append(sex)
                lick_dict_probe['session'].append(np.unique(behav['session'])[0])
                lick_dict_probe['num_licks'].append(len(behav_probe[behav_probe['lick_port'] != -1]))
                lick_dict_probe['probe_pc'].append(pc)
                lick_dict_probe['session_pc'].append(session_pc)
            else:
                pass
## Convert to dataframe
probe_df = pd.DataFrame(lick_dict_probe)
avg_probe = probe_df.groupby(['group', 'day'], as_index=False).agg({'probe_pc': ['mean', 'sem']})
avg_probe_mf = probe_df.groupby(['group', 'sex', 'day'], as_index=False).agg({'probe_pc': ['mean', 'sem']})

In [ ]:
behav

In [ ]:
## Plot average probe performance
fig = pf.custom_graph_template(x_title='Day', y_title='Probe Accuracy', width=600)
for group in np.unique(avg_probe['group']):
    group_data = avg_probe[avg_probe['group'] == group]
    fig.add_trace(go.Scatter(x=group_data['day'], y=group_data['probe_pc']['mean'], name=group, mode='lines+markers',
                            line_color=group_colors_dict[group], error_y=dict(type='data', array=group_data['probe_pc']['sem'])))
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_vline(x=12.5, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.show() 
fig.write_image(pjoin(fig_path, 'probe_accuracy.png'))

In [ ]:
## Plot average probe performance split by male female
fig = pf.custom_graph_template(x_title='Day', y_title='Probe Accuracy', width=600)
for group in np.unique(avg_probe_mf['group']):
    group_data = avg_probe_mf[avg_probe_mf['group'] == group]
    for sex in np.unique(avg_probe_mf['sex']):
        if (group == 'WT') & (sex == 'Male'):
            key = 'Wildtype Male'
        elif (group == 'APP-KI') & (sex == 'Male'):
            key = 'APP-KI Male' 
        elif (group == 'WT') & (sex == 'Female'):
            key = 'Wildtype Female' 
        else:
            key = 'APP-KI Female'
        plot_data = group_data[group_data['sex'] == sex]
        fig.add_trace(go.Scatter(x=plot_data['day'], y=plot_data['probe_pc']['mean'], name=key, mode='lines+markers',
                                line_color=group_colors_dict[key], error_y=dict(type='data', array=plot_data['probe_pc']['sem'])))
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_vline(x=12.5, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.show() 
fig.write_image(pjoin(fig_path, 'probe_accuracy_mf.png'))

### Analyze whether mice were licking at previously rewarded ports during the probe on the first reversal day.

In [ ]:
lick_dict_probe = {'mouse': [], 'day': [], 'group': [], 'sex': [], 'session': [], 'num_licks': [], 'probe_pc': [], 'session_pc': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass
    else:
        mouse_path = pjoin(circle_path, mouse)
        group = 'WT' if mouse in wt_mice else 'APP-KI'
        sex = 'Male' if mouse in male_mice else 'Female'
        for idx, session in enumerate(os.listdir(mouse_path)):
            behav = pd.read_feather(pjoin(mouse_path, f'{session}'))
            if (behav['session'].to_numpy()[0] == 'AP'):
                behav_probe = behav.loc[behav['probe']]
                behav_no_probe = behav[~behav['probe']]
                reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
                ## Percent correct licking
                pc = ctb.lick_accuracy(behav_probe, port_one=reward_one, port_two=reward_two, by_trials=False)
                session_pc = ctb.lick_accuracy(behav_no_probe, port_one=reward_one, port_two=reward_two, by_trials=False)
                lick_dict_probe['mouse'].append(mouse)
                lick_dict_probe['day'].append(idx+1)
                lick_dict_probe['group'].append(group)
                lick_dict_probe['sex'].append(sex)
                lick_dict_probe['session'].append(np.unique(behav['session'])[0])
                lick_dict_probe['num_licks'].append(len(behav_probe[behav_probe['lick_port'] != -1]))
                lick_dict_probe['probe_pc'].append(pc)
                lick_dict_probe['session_pc'].append(session_pc)
            elif (behav['session'].to_numpy()[0] == 'RP'):
                behav_probe = behav.loc[behav['probe']]
                behav_no_probe = behav[~behav['probe']]
                ## Percent correct licking using the reward ports from the previous probe session (before ports switched)
                pc = ctb.lick_accuracy(behav_probe, port_one=reward_one, port_two=reward_two, by_trials=False)
                session_pc = ctb.lick_accuracy(behav_no_probe, port_one=reward_one, port_two=reward_two, by_trials=False)
                lick_dict_probe['mouse'].append(mouse)
                lick_dict_probe['day'].append(idx+1)
                lick_dict_probe['group'].append(group)
                lick_dict_probe['sex'].append(sex)
                lick_dict_probe['session'].append(np.unique(behav['session'])[0])
                lick_dict_probe['num_licks'].append(len(behav_probe[behav_probe['lick_port'] != -1]))
                lick_dict_probe['probe_pc'].append(pc)
                lick_dict_probe['session_pc'].append(session_pc)
            else:
                pass
## Convert to dataframe
probe_df = pd.DataFrame(lick_dict_probe)
avg_probe = probe_df.groupby(['group', 'day'], as_index=False).agg({'probe_pc': ['mean', 'sem']})
avg_probe_mf = probe_df.groupby(['group', 'sex', 'day'], as_index=False).agg({'probe_pc': ['mean', 'sem']})

In [ ]:
## Plot average probe performance with the previoiusly rewarded ports used after reversal
fig = pf.custom_graph_template(x_title='Day', y_title='Probe Accuracy', width=600)
for group in np.unique(avg_probe['group']):
    group_data = avg_probe[avg_probe['group'] == group]
    fig.add_trace(go.Scatter(x=group_data['day'], y=group_data['probe_pc']['mean'], name=group, mode='lines+markers',
                            line_color=group_colors_dict[group], error_y=dict(type='data', array=group_data['probe_pc']['sem'])))
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_vline(x=12.5, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.show() 
fig.write_image(pjoin(fig_path, 'probe_accuracy_previous_ports.png'))

In [ ]:
## Plot average probe performance split by male female
fig = pf.custom_graph_template(x_title='Day', y_title='Probe Accuracy', width=600)
for group in np.unique(avg_probe_mf['group']):
    group_data = avg_probe_mf[avg_probe_mf['group'] == group]
    for sex in np.unique(avg_probe_mf['sex']):
        if (group == 'WT') & (sex == 'Male'):
            key = 'Wildtype Male'
        elif (group == 'APP-KI') & (sex == 'Male'):
            key = 'APP-KI Male' 
        elif (group == 'WT') & (sex == 'Female'):
            key = 'Wildtype Female' 
        else:
            key = 'APP-KI Female'
        plot_data = group_data[group_data['sex'] == sex]
        fig.add_trace(go.Scatter(x=plot_data['day'], y=plot_data['probe_pc']['mean'], name=key, mode='lines+markers',
                                line_color=group_colors_dict[key], error_y=dict(type='data', array=plot_data['probe_pc']['sem'])))
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_vline(x=12.5, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.show() 
fig.write_image(pjoin(fig_path, 'probe_accuracy_mf_previous_ports.png'))

### Analyze behavior across trials for WT and APP-KI mice.

In [ ]:
## Circle track accuracy across trials
## Plots individual mice
day_of_interest = 13
bin_size = 4
opacity = 0.8
fig = pf.custom_graph_template(x_title='', y_title='', height=500, width=600,
                               rows=1, columns=2, shared_x=True, shared_y=True,
                               titles=['Wildtype', 'APP-KI'])
for mouse in os.listdir(circle_path):
    mouse_path = pjoin(circle_path, mouse)
    group = 'WT' if mouse in wt_mice else 'APP-KI'
    for session in os.listdir(mouse_path):
        if f'_{day_of_interest}' in session:
            behav_data = pd.read_feather(pjoin(mouse_path, f'{session}'))
            reward_one, reward_two = np.unique(behav_data['reward_one'])[0], np.unique(behav_data['reward_two'])[0]
            pc = ctb.lick_accuracy(behav_data, reward_one, reward_two, by_trials=True)
            binned_pc = ctb.bin_data(pc, bin_size=bin_size)
            x_data = np.arange(1, len(binned_pc)+1) * bin_size
            if group == 'WT':
                col = 1
            else:
                col = 2
            fig.add_trace(go.Scatter(x=x_data, y=binned_pc, mode='lines', opacity=opacity,
                                    line_color=group_colors_dict[group], showlegend=False, name=mouse), row=1, col=col)
        else:
            pass
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(title='Lick Accuracy', col=1)
fig.update_xaxes(title='Trial', row=1)
fig.show()

In [ ]:
## Circle track accuracy across trials
## Plots individual mice
day_of_interest = 5
bin_size = 4
opacity = 0.8
fig = pf.custom_graph_template(x_title='', y_title='', height=500, width=600,
                               rows=1, columns=2, shared_x=True, shared_y=True,
                               titles=['Wildtype', 'APP-KI'])
for mouse in os.listdir(circle_path):
    mouse_path = pjoin(circle_path, mouse)
    group = 'WT' if mouse in wt_mice else 'APP-KI'
    for session in os.listdir(mouse_path):
        if f'_{day_of_interest}' in session:
            behav_data = pd.read_feather(pjoin(mouse_path, f'{session}'))
            reward_one, reward_two = np.unique(behav_data['reward_one'])[0], np.unique(behav_data['reward_two'])[0]
            pc = ctb.lick_accuracy(behav_data, reward_one, reward_two, by_trials=True)
            binned_pc = ctb.bin_data(pc, bin_size=bin_size)
            x_data = np.arange(1, len(binned_pc)+1) * bin_size
            if group == 'WT':
                col = 1
            else:
                col = 2
            fig.add_trace(go.Scatter(x=x_data, y=binned_pc, mode='lines', opacity=opacity,
                                    line_color=group_colors_dict[group], showlegend=False, name=mouse), row=1, col=col)
        else:
            pass
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(title='Lick Accuracy', col=1)
fig.update_xaxes(title='Trial', row=1)
fig.show()

### Look at lick accuracy up to the minimum number of trials across all mice for all days split into four groups.

In [ ]:
## Average circle track accuracy up to a minimum number of trials for both groups
bin_size = 4
chance_color = 'darkgrey'
opacity = 0.8
data = {'mouse': [], 'day': [], 'group': [], 'sex': [], 'trial_acc': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass
    else:
        mouse_path = pjoin(circle_path, mouse)
        group = 'WT' if mouse in wt_mice else 'APP-KI' 
        sex = 'Male' if mouse in male_mice else 'Female'
        for idx, session in enumerate(os.listdir(mouse_path)):
            behav_data = pd.read_feather(pjoin(mouse_path, f'{session}'))
            reward_one, reward_two = np.unique(behav_data['reward_one'])[0], np.unique(behav_data['reward_two'])[0]
            pc = ctb.lick_accuracy(behav_data, reward_one, reward_two, by_trials=True)
            binned_pc = ctb.bin_data(pc, bin_size=bin_size)
            x_data = np.arange(1, len(binned_pc) + 1) * bin_size
            data['mouse'].append(mouse)
            data['day'].append(idx + 1)
            data['group'].append(group)
            data['sex'].append(sex)
            data['trial_acc'].append(binned_pc)

trial_acc = pd.DataFrame(data)
group_data = {'Wildtype Male': [], 'APP-KI Male': [], 'Wildtype Female': [], 'APP-KI Female': [], 'day': []}
for day in np.unique(trial_acc['day']):
    group_data['Wildtype Male'].append(np.nanmean(trial_acc['trial_acc'][(trial_acc['group'] == 'WT') & (trial_acc['sex'] == 'Male') & (trial_acc['day'] == day)].apply(pd.Series), axis=0))
    group_data['APP-KI Male'].append(np.nanmean(trial_acc['trial_acc'][(trial_acc['group'] == 'APP-KI') & (trial_acc['sex'] == 'Male') & (trial_acc['day'] == day)].apply(pd.Series), axis=0))
    group_data['Wildtype Female'].append(np.nanmean(trial_acc['trial_acc'][(trial_acc['group'] == 'WT') & (trial_acc['sex'] == 'Female') & (trial_acc['day'] == day)].apply(pd.Series), axis=0))
    group_data['APP-KI Female'].append(np.nanmean(trial_acc['trial_acc'][(trial_acc['group'] == 'APP-KI') & (trial_acc['sex'] == 'Female') & (trial_acc['day'] == day)].apply(pd.Series), axis=0))
    group_data['day'].append(day)
group_df = pd.DataFrame(group_data)

In [ ]:
only_min_trials = False
fig = pf.custom_graph_template(x_title='', y_title='', height=1000, width=800,
                               rows=5, columns=4, shared_x=True, shared_y=True,
                               titles=[f'Day {x}' for x in np.arange(1, len(group_df['day'])+1)])
for idx, day in enumerate(group_df['day']):
    plot_data = group_df[group_df['day'] == day]

    if only_min_trials:
        trial_lengths = []
        for group in plot_data.columns[:-1]:
            trial_lengths.append(len(plot_data.loc[:, group].values[0]))
        min_trials = np.min(trial_lengths)
    else:
        min_trials = -1
    
    for group in plot_data.columns[:-1]:
        group_data = plot_data.loc[:, group].values[0][:min_trials]
        x_data = np.arange(1, len(group_data)+1) * bin_size
        if idx < 4:
            row, col = 1, idx + 1
        elif (idx >= 4) & (idx < 8):
            row, col = 2, idx - 3
        elif (idx >= 8) & (idx < 12):
            row, col = 3, idx - 7
        elif (idx >= 12) & (idx < 16):
            row, col = 4, idx - 11
        elif (idx >= 16) & (idx < 20):
            row, col = 5, idx - 15
        fig.add_trace(go.Scatter(x=x_data, y=group_data, mode='lines', opacity=opacity,
                                line_color=group_colors_dict[group], showlegend=False,
                                legendgroup=group, name=group), row=row, col=col)
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(title='Lick Accuracy', col=1)
fig.update_xaxes(title='Trial', row=5)
fig.show()
if only_min_trials:
    fig.write_image(pjoin(fig_path, 'lick_acc_min_trials_across_groups.png'))
else:
    fig.write_image(pjoin(fig_path, 'lick_acc_across_groups.png'))

In [ ]:
## Average circle track accuracy up to a minimum number of trials for both groups
bin_size = 4
chance_color = 'darkgrey'
opacity = 0.8
data = {'mouse': [], 'day': [], 'group': [], 'trial_acc': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass
    else:
        mouse_path = pjoin(circle_path, mouse)
        group = 'WT' if mouse in wt_mice else 'APP-KI' 
        for idx, session in enumerate(os.listdir(mouse_path)):
            behav_data = pd.read_feather(pjoin(mouse_path, f'{session}'))
            reward_one, reward_two = np.unique(behav_data['reward_one'])[0], np.unique(behav_data['reward_two'])[0]
            pc = ctb.lick_accuracy(behav_data, reward_one, reward_two, by_trials=True)
            binned_pc = ctb.bin_data(pc, bin_size=bin_size)
            x_data = np.arange(1, len(binned_pc) + 1) * bin_size
            data['mouse'].append(mouse)
            data['day'].append(idx + 1)
            data['group'].append(group)
            data['trial_acc'].append(binned_pc)

trial_acc = pd.DataFrame(data)
group_data = {'Wildtype': [], 'APP-KI': [], 'day': []}
for day in np.unique(trial_acc['day']):
    group_data['Wildtype'].append(np.nanmean(trial_acc['trial_acc'][(trial_acc['group'] == 'WT') & (trial_acc['day'] == day)].apply(pd.Series), axis=0))
    group_data['APP-KI'].append(np.nanmean(trial_acc['trial_acc'][(trial_acc['group'] == 'APP-KI') & (trial_acc['day'] == day)].apply(pd.Series), axis=0))
    group_data['Wildtype'].append(np.std(trial_acc['trial_acc'][(trial_acc['group'] == 'WT') & (trial_acc['day'] == day)].apply(pd.Series), axis=0, ddof=1).to_numpy() / np.sqrt(len(trial_acc['trial_acc'][(trial_acc['group'] == 'WT') & (trial_acc['day'] == day)].apply(pd.Series))))
    group_data['APP-KI'].append(np.std(trial_acc['trial_acc'][(trial_acc['group'] == 'APP-KI') & (trial_acc['day'] == day)].apply(pd.Series), axis=0, ddof=1).to_numpy() / np.sqrt(len(trial_acc['trial_acc'][(trial_acc['group'] == 'APP-KI') & (trial_acc['day'] == day)].apply(pd.Series))))
    group_data['day'].append(day)
    group_data['day'].append(day)
group_df = pd.DataFrame(group_data)

In [ ]:
only_min_trials = False
error_color = 'rgba(169, 169, 169, 0.4)' ## rgba is the only way to get the error band transparent
fig = pf.custom_graph_template(x_title='', y_title='', height=1200, width=800,
                               rows=6, columns=4, shared_x=True, shared_y=True,
                               titles=[f'Day {x}' for x in np.arange(1, len(np.unique(group_df['day']))+1)])
for idx, day in enumerate(np.unique(group_df['day'])):
    plot_data = group_df[group_df['day'] == day]

    if only_min_trials:
        trial_lengths = []
        for group in plot_data.columns[:-1]:
            trial_lengths.append(len(plot_data.loc[:, group].values[0]))
        min_trials = np.min(trial_lengths)
    else:
        min_trials = -1
    
    for group in plot_data.columns[:-1]:
        group_data = plot_data.loc[:, group].values[0][:min_trials]
        upper = plot_data.loc[:, group].values[0][:min_trials] + plot_data.loc[:, group].values[1][:min_trials]
        lower = plot_data.loc[:, group].values[0][:min_trials] - plot_data.loc[:, group].values[1][:min_trials]
        x_data = np.arange(1, len(group_data)+1) * bin_size
        if idx < 4:
            row, col = 1, idx + 1
        elif (idx >= 4) & (idx < 8):
            row, col = 2, idx - 3
        elif (idx >= 8) & (idx < 12):
            row, col = 3, idx - 7
        elif (idx >= 12) & (idx < 16):
            row, col = 4, idx - 11
        elif (idx >= 16) & (idx < 20):
            row, col = 5, idx - 15
        elif (idx >= 20) & (idx < 24):
            row, col = 6, idx - 19
        fig.add_trace(go.Scatter(x=x_data, y=group_data, mode='lines', opacity=opacity,
                                line_color=group_colors_dict[group], showlegend=False,
                                legendgroup=group, name=group), row=row, col=col)
        fig.add_trace(go.Scatter(name='Upper Bound', x=x_data, y=upper, mode='lines',
                                 marker=dict(color=error_color), line=dict(width=0), showlegend=False, 
                                 legendgroup=group), row=row, col=col)
        fig.add_trace(go.Scatter(name='Lower Bound', x=x_data, y=lower, marker=dict(color=error_color),
                                 line=dict(width=0), mode='lines', fillcolor=error_color, fill='tonexty', 
                                 showlegend=False, legendgroup=group), row=row, col=col)
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(title='Lick Accuracy', range=[0, 100], dtick=25, col=1)
fig.update_xaxes(title='Trial', row=6)
fig.show()
if only_min_trials:
    fig.write_image(pjoin(fig_path, 'lick_acc_min_trials_across_groups_wtapp.png'))
else:
    fig.write_image(pjoin(fig_path, 'lick_acc_across_groups_wtapp.png'))

In [ ]:
## Plot lick accuracy across the trial for a specific day
only_min_trials = True
day = 14
# error_color = 'rgba(169, 169, 169, 0.4)' ## rgba is the only way to get the error band transparent
error_colors = ['rgba(25,25,112,0.4)', 'rgba(255,0,0,0.4)']
fig = pf.custom_graph_template(x_title='', y_title='', height=500, width=500,
                               titles=['Second Day Reversal'])
plot_data = group_df[group_df['day'] == day]

if only_min_trials:
        trial_lengths = []
        for group in plot_data.columns[:-1]:
            trial_lengths.append(len(plot_data.loc[:, group].values[0]))
        min_trials = np.min(trial_lengths)
else:
    min_trials = -1

for idx, group in enumerate(plot_data.columns[:-1]):
        group_data = plot_data.loc[:, group].values[0][:min_trials]
        upper = plot_data.loc[:, group].values[0][:min_trials] + plot_data.loc[:, group].values[1][:min_trials]
        lower = plot_data.loc[:, group].values[0][:min_trials] - plot_data.loc[:, group].values[1][:min_trials]
        x_data = np.arange(1, len(group_data)+1) * bin_size
        fig.add_trace(go.Scatter(x=x_data, y=group_data, mode='lines', opacity=opacity,
                                line_color=group_colors_dict[group], showlegend=False,
                                legendgroup=group, name=group))
        fig.add_trace(go.Scatter(name='Upper Bound', x=x_data, y=upper, mode='lines',
                                 marker=dict(color=error_colors[idx]), line=dict(width=0), showlegend=False, 
                                 legendgroup=group))
        fig.add_trace(go.Scatter(name='Lower Bound', x=x_data, y=lower, marker=dict(color=error_colors[idx]),
                                 line=dict(width=0), mode='lines', fillcolor=error_colors[idx], fill='tonexty', 
                                 showlegend=False, legendgroup=group))
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(title='Lick Accuracy', range=[0, 100], dtick=25, col=1)
fig.update_xaxes(title='Trial')
fig.show()
# if only_min_trials:
#     fig.write_image(pjoin(fig_path, f'lick_acc_min_trials_across_groups_wtapp_day{day}.png'))
# else:
#     fig.write_image(pjoin(fig_path, f'lick_acc_across_groups_wtapp_day{day}.png'))
# fig.write_image(pjoin(fig_path, 'last_day_training_wtapp_mintrials.png'))
fig.write_image(pjoin(fig_path, 'second_day_reversal_wtapp_mintrials.svg'))

In [ ]:
day_of_interest = 12
anova_dict = {'mouse': [], 'genotype': [], 'day': [], 'trial': [], 'accuracy': []}
for mouse in trial_acc['mouse']:
    loop_data = trial_acc[(trial_acc['mouse'] == mouse) & (trial_acc['day'] == day_of_interest)]
    x_data = np.arange(1, len(loop_data['trial_acc'].values[0])+1) * bin_size
    if loop_data['group'].values[0] == 'WT':
        genotype = 'Wildtype'
    elif loop_data['group'].values[0] == 'APP-KI':
        genotype = 'APP-KI'
    for idx, value in enumerate(loop_data['trial_acc'].values[0]):
        anova_dict['mouse'].append(mouse)
        anova_dict['genotype'].append(genotype)
        anova_dict['trial'].append(x_data[idx])
        anova_dict['accuracy'].append(value)
anova_data = pd.DataFrame(anova_dict)
# anova_data = anova_data[~pd.isna(anova_data['accuracy'])]
# anova_data.mixed_anova(dv='accuracy', within='trial', subject='mouse', between='genotype')

### Concatenate lick accuracy for all trials across all days for all four groups.

In [ ]:
bin_size = 5
data = {'mouse': [], 'group': [], 'sex': [], 'trial_acc': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass
    else:
        mouse_path = pjoin(circle_path, mouse)
        group = 'WT' if mouse in wt_mice else 'APP-KI' 
        sex = 'Male' if mouse in male_mice else 'Female'
        accuracy = []
        for idx, session in enumerate(os.listdir(mouse_path)):
            behav_data = pd.read_feather(pjoin(mouse_path, f'{session}'))
            reward_one, reward_two = np.unique(behav_data['reward_one'])[0], np.unique(behav_data['reward_two'])[0]
            pc = ctb.lick_accuracy(behav_data, reward_one, reward_two, by_trials=True)
            accuracy = np.concatenate((accuracy, pc))
        data['mouse'].append(mouse)
        data['group'].append(group)
        data['sex'].append(sex)
        data['trial_acc'].append(accuracy)
        
trial_data = pd.DataFrame(data)
data['binned_pc'] = []
for idx, mouse in enumerate(trial_data['mouse']):
    d = trial_data['trial_acc'][trial_data['mouse'] == mouse]
    data['binned_pc'].append(ctb.bin_data(d.values[0], bin_size=bin_size))
trial_data = pd.DataFrame(data)
group_data = {'Wildtype Male': [], 'APP-KI Male': [], 'Wildtype Female': [], 'APP-KI Female': []}
group_data['Wildtype Male'].append(np.nanmean(trial_data['binned_pc'][(trial_data['group'] == 'WT') & (trial_data['sex'] == 'Male')].apply(pd.Series), axis=0))
group_data['APP-KI Male'].append(np.nanmean(trial_data['binned_pc'][(trial_data['group'] == 'APP-KI') & (trial_data['sex'] == 'Male')].apply(pd.Series), axis=0))
group_data['Wildtype Female'].append(np.nanmean(trial_data['binned_pc'][(trial_data['group'] == 'WT') & (trial_data['sex'] == 'Female')].apply(pd.Series), axis=0))
group_data['APP-KI Female'].append(np.nanmean(trial_data['binned_pc'][(trial_data['group'] == 'APP-KI') & (trial_data['sex'] == 'Female')].apply(pd.Series), axis=0))
group_data['Wildtype Male'].append(np.std(trial_data['binned_pc'][(trial_data['group'] == 'WT') & (trial_data['sex'] == 'Male')].apply(pd.Series), axis=0, ddof=1).to_numpy() / np.sqrt(len(trial_data['binned_pc'][(trial_data['group'] == 'WT') & (trial_data['sex'] == 'Male')].apply(pd.Series))))
group_data['APP-KI Male'].append(np.std(trial_data['binned_pc'][(trial_data['group'] == 'APP-KI') & (trial_data['sex'] == 'Male')].apply(pd.Series), axis=0, ddof=1).to_numpy() / np.sqrt(len(trial_data['binned_pc'][(trial_data['group'] == 'APP-KI') & (trial_data['sex'] == 'Male')].apply(pd.Series))))
group_data['Wildtype Female'].append(np.std(trial_data['binned_pc'][(trial_data['group'] == 'WT') & (trial_data['sex'] == 'Female')].apply(pd.Series), axis=0, ddof=1).to_numpy() / np.sqrt(len(trial_data['binned_pc'][(trial_data['group'] == 'WT') & (trial_data['sex'] == 'Female')].apply(pd.Series))))
group_data['APP-KI Female'].append(np.std(trial_data['binned_pc'][(trial_data['group'] == 'APP-KI') & (trial_data['sex'] == 'Female')].apply(pd.Series), axis=0, ddof=1).to_numpy() / np.sqrt(len(trial_data['binned_pc'][(trial_data['group'] == 'APP-KI') & (trial_data['sex'] == 'Female')].apply(pd.Series))))
plot_data = pd.DataFrame(group_data)

In [ ]:
data_points = 20 ## the number of trials plotted will be this times the bin size used
error_color = 'rgba(169, 169, 169, 0.4)' ## rgba is the only way to get the error band transparent
fig = pf.custom_graph_template(x_title='Trial', y_title='Lick Accuracy (%)', height=600, width=700)
for group in plot_data.columns:
    group_data = plot_data.loc[:, group]
    x_data = np.arange(1, len(group_data[0])+1) * bin_size
    upper = group_data[0] + group_data[1]
    lower = group_data[0] - group_data[1]
    fig.add_trace(go.Scatter(name='Upper Bound', x=x_data[:data_points], y=upper[:data_points], mode='lines',
        marker=dict(color=error_color), line=dict(width=0), showlegend=False, legendgroup=group, opacity=0.7))
    fig.add_trace(go.Scatter(name='Lower Bound', x=x_data[:data_points], y=lower[:data_points], marker=dict(color=error_color),
        line=dict(width=0), mode='lines', fillcolor=error_color, fill='tonexty', showlegend=False, legendgroup=group))
    fig.add_trace(go.Scatter(x=x_data[:data_points], y=group_data[0][:data_points], mode='lines', name=group,
                             line_color=group_colors_dict[group], legendgroup=group))
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.show()

In [ ]:
idx_num = 20 ## mixed anova requires that all subjects have the same number of trials
anova_dict = {'mouse': [], 'sex': [], 'genotype': [], 'trial': [], 'accuracy': []}
for mouse in trial_data['mouse']:
    loop_data = trial_data[trial_data['mouse'] == mouse]
    x_data = np.arange(1, len(loop_data['binned_pc'].values[0][:idx_num])+1) * bin_size
    if (loop_data['group'].values[0] == 'WT') & (loop_data['sex'].values[0] == 'Male'):
        sex = 'Male' 
        genotype = 'Wildtype'
    elif (loop_data['group'].values[0] == 'APP-KI') & (loop_data['sex'].values[0] == 'Male'):
        sex = 'Male'
        genotype = 'APP-KI'
    elif (loop_data['group'].values[0] == 'WT') & (loop_data['sex'].values[0] == 'Female'):
        sex = 'Female'
        genotype = 'Wildtype'
    else:
        sex = 'Female'
        genotype = 'APP-KI'
    for idx, value in enumerate(loop_data['binned_pc'].values[0][:idx_num]):
        anova_dict['mouse'].append(mouse)
        anova_dict['sex'].append(sex)
        anova_dict['genotype'].append(genotype)
        anova_dict['trial'].append(x_data[idx])
        anova_dict['accuracy'].append(value)
anova_data = pd.DataFrame(anova_dict)
anova_data = anova_data[~pd.isna(anova_data['accuracy'])]
anova_data.mixed_anova(dv='accuracy', within='trial', subject='mouse', between='genotype')

### Concatenate lick accuracy across trials for all days for wildtype vs APP-KI.

In [ ]:
bin_size = 5
data = {'mouse': [], 'group': [], 'sex': [], 'trial_acc': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass
    else:
        mouse_path = pjoin(circle_path, mouse)
        group = 'WT' if mouse in wt_mice else 'APP-KI' 
        sex = 'Male' if mouse in male_mice else 'Female'
        accuracy = []
        for idx, session in enumerate(os.listdir(mouse_path)):
            behav_data = pd.read_feather(pjoin(mouse_path, f'{session}'))
            reward_one, reward_two = np.unique(behav_data['reward_one'])[0], np.unique(behav_data['reward_two'])[0]
            pc = ctb.lick_accuracy(behav_data, reward_one, reward_two, by_trials=True)
            accuracy = np.concatenate((accuracy, pc))
        data['mouse'].append(mouse)
        data['group'].append(group)
        data['sex'].append(sex)
        data['trial_acc'].append(accuracy)
        
trial_data = pd.DataFrame(data)
data['binned_pc'] = []
for idx, mouse in enumerate(trial_data['mouse']):
    d = trial_data['trial_acc'][trial_data['mouse'] == mouse]
    data['binned_pc'].append(ctb.bin_data(d.values[0], bin_size=bin_size))
trial_data = pd.DataFrame(data)
group_data = {'Wildtype': [], 'APP-KI': []}
group_data['Wildtype'].append(np.nanmean(trial_data['binned_pc'][trial_data['group'] == 'WT'].apply(pd.Series), axis=0))
group_data['APP-KI'].append(np.nanmean(trial_data['binned_pc'][trial_data['group'] == 'APP-KI'].apply(pd.Series), axis=0))
group_data['Wildtype'].append(np.std(trial_data['binned_pc'][trial_data['group'] == 'WT'].apply(pd.Series), axis=0, ddof=1).to_numpy() / np.sqrt(len(trial_data['binned_pc'][trial_data['group'] == 'WT'].apply(pd.Series))))
group_data['APP-KI'].append(np.std(trial_data['binned_pc'][trial_data['group'] == 'APP-KI'].apply(pd.Series), axis=0, ddof=1).to_numpy() / np.sqrt(len(trial_data['binned_pc'][trial_data['group'] == 'APP-KI'].apply(pd.Series))))
plot_data = pd.DataFrame(group_data)

In [ ]:
data_points = 20 ## the number of trials plotted will be this times the bin size used
error_color = 'rgba(169, 169, 169, 0.4)' ## rgba is the only way to get the error band transparent
fig = pf.custom_graph_template(x_title='Trial', y_title='Lick Accuracy (%)', height=600, width=700, titles=['Training'])
for group in plot_data.columns:
    group_data = plot_data.loc[:, group]
    x_data = np.arange(1, len(group_data[0])+1) * bin_size
    upper = group_data[0] + group_data[1]
    lower = group_data[0] - group_data[1]
    fig.add_trace(go.Scatter(name='Upper Bound', x=x_data[:data_points], y=upper[:data_points], mode='lines',
        marker=dict(color=error_color), line=dict(width=0), showlegend=False, legendgroup=group, opacity=0.7))
    fig.add_trace(go.Scatter(name='Lower Bound', x=x_data[:data_points], y=lower[:data_points], marker=dict(color=error_color),
        line=dict(width=0), mode='lines', fillcolor=error_color, fill='tonexty', showlegend=False, legendgroup=group))
    fig.add_trace(go.Scatter(x=x_data[:data_points], y=group_data[0][:data_points], mode='lines', name=group,
                             line_color=group_colors_dict[group], legendgroup=group))
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(range=[0, 100], dtick=25)
fig.show()
fig.write_image(pjoin(fig_path, f'concatenated_{data_points*bin_size}trials_wt_app_training.png'))

In [ ]:
idx_num = 20 ## mixed anova requires that all subjects have the same number of trials
anova_dict = {'mouse': [], 'sex': [], 'genotype': [], 'trial': [], 'accuracy': []}
for mouse in trial_data['mouse']:
    loop_data = trial_data[trial_data['mouse'] == mouse]
    x_data = np.arange(1, len(loop_data['binned_pc'].values[0][:idx_num])+1) * bin_size
    if loop_data['group'].values[0] == 'WT':
        genotype = 'Wildtype'
    elif loop_data['group'].values[0] == 'APP-KI':
        genotype = 'APP-KI'
    for idx, value in enumerate(loop_data['binned_pc'].values[0][:idx_num]):
        anova_dict['mouse'].append(mouse)
        anova_dict['sex'].append(sex)
        anova_dict['genotype'].append(genotype)
        anova_dict['trial'].append(x_data[idx])
        anova_dict['accuracy'].append(value)
anova_data = pd.DataFrame(anova_dict)
anova_data = anova_data[~pd.isna(anova_data['accuracy'])]
anova_data.mixed_anova(dv='accuracy', within='trial', subject='mouse', between='genotype')

### Concatenate lick accuracy across trials starting from the reversal.

In [ ]:
bin_size = 5
data = {'mouse': [], 'group': [], 'sex': [], 'trial_acc': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass
    else:
        mouse_path = pjoin(circle_path, mouse)
        group = 'WT' if mouse in wt_mice else 'APP-KI' 
        sex = 'Male' if mouse in male_mice else 'Female'
        accuracy = []
        for idx, session in enumerate(os.listdir(mouse_path)):
            behav_data = pd.read_feather(pjoin(mouse_path, f'{session}'))
            if (behav_data['session'].to_numpy()[0] == 'RP') | (behav_data['session'].to_numpy()[0] == 'AR'):
                reward_one, reward_two = np.unique(behav_data['reward_one'])[0], np.unique(behav_data['reward_two'])[0]
                pc = ctb.lick_accuracy(behav_data, reward_one, reward_two, by_trials=True)
                accuracy = np.concatenate((accuracy, pc))
            else:
                pass
        data['mouse'].append(mouse)
        data['group'].append(group)
        data['sex'].append(sex)
        data['trial_acc'].append(accuracy)
            
trial_data = pd.DataFrame(data)
data['binned_pc'] = []
for idx, mouse in enumerate(trial_data['mouse']):
    d = trial_data['trial_acc'][trial_data['mouse'] == mouse]
    data['binned_pc'].append(ctb.bin_data(d.values[0], bin_size=bin_size))
trial_data = pd.DataFrame(data)
group_data = {'Wildtype': [], 'APP-KI': []}
group_data['Wildtype'].append(np.nanmean(trial_data['binned_pc'][trial_data['group'] == 'WT'].apply(pd.Series), axis=0))
group_data['APP-KI'].append(np.nanmean(trial_data['binned_pc'][trial_data['group'] == 'APP-KI'].apply(pd.Series), axis=0))
group_data['Wildtype'].append(np.std(trial_data['binned_pc'][trial_data['group'] == 'WT'].apply(pd.Series), axis=0, ddof=1).to_numpy() / np.sqrt(len(trial_data['binned_pc'][trial_data['group'] == 'WT'].apply(pd.Series))))
group_data['APP-KI'].append(np.std(trial_data['binned_pc'][trial_data['group'] == 'APP-KI'].apply(pd.Series), axis=0, ddof=1).to_numpy() / np.sqrt(len(trial_data['binned_pc'][trial_data['group'] == 'APP-KI'].apply(pd.Series))))
plot_data = pd.DataFrame(group_data)

In [ ]:
data_points = 20 ## the number of trials plotted will be this times the bin size used
error_color = 'rgba(169, 169, 169, 0.4)' ## rgba is the only way to get the error band transparent
fig = pf.custom_graph_template(x_title='Trial', y_title='Lick Accuracy (%)', height=600, width=700, titles=['Reversal'])
for group in plot_data.columns:
    group_data = plot_data.loc[:, group]
    x_data = np.arange(1, len(group_data[0])+1) * bin_size
    upper = group_data[0] + group_data[1]
    lower = group_data[0] - group_data[1]
    fig.add_trace(go.Scatter(name='Upper Bound', x=x_data[:data_points], y=upper[:data_points], mode='lines',
        marker=dict(color=error_color), line=dict(width=0), showlegend=False, legendgroup=group, opacity=0.7))
    fig.add_trace(go.Scatter(name='Lower Bound', x=x_data[:data_points], y=lower[:data_points], marker=dict(color=error_color),
        line=dict(width=0), mode='lines', fillcolor=error_color, fill='tonexty', showlegend=False, legendgroup=group))
    fig.add_trace(go.Scatter(x=x_data[:data_points], y=group_data[0][:data_points], mode='lines', name=group,
                             line_color=group_colors_dict[group], legendgroup=group))
fig.add_hline(y=75, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.add_hline(y=25, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(range=[0, 100], dtick=25)
fig.show()
fig.write_image(pjoin(fig_path, f'concatenated_{data_points*bin_size}trials_wt_app_reversal.png'))

In [ ]:
idx_num = 20 ## mixed anova requires that all subjects have the same number of trials
anova_dict = {'mouse': [], 'sex': [], 'genotype': [], 'trial': [], 'accuracy': []}
for mouse in trial_data['mouse']:
    loop_data = trial_data[trial_data['mouse'] == mouse]
    x_data = np.arange(1, len(loop_data['binned_pc'].values[0][:idx_num])+1) * bin_size
    if loop_data['group'].values[0] == 'WT':
        genotype = 'Wildtype'
    elif loop_data['group'].values[0] == 'APP-KI':
        genotype = 'APP-KI'
    for idx, value in enumerate(loop_data['binned_pc'].values[0][:idx_num]):
        anova_dict['mouse'].append(mouse)
        anova_dict['sex'].append(sex)
        anova_dict['genotype'].append(genotype)
        anova_dict['trial'].append(x_data[idx])
        anova_dict['accuracy'].append(value)
anova_data = pd.DataFrame(anova_dict)
anova_data = anova_data[~pd.isna(anova_data['accuracy'])]
anova_data.mixed_anova(dv='accuracy', within='trial', subject='mouse', between='genotype')

### Signal detection metrics across days for all mice.

In [ ]:
metrics = pd.DataFrame()
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass
    else:
        mouse_path = pjoin(circle_path, mouse)
        group = 'WT' if mouse in wt_mice else 'APP-KI' 
        sex = 'Male' if mouse in male_mice else 'Female'
        for idx, session in enumerate(os.listdir(mouse_path)):
            behav_data = pd.read_feather(pjoin(mouse_path, f'{session}'))
            reward_one, reward_two = np.unique(behav_data['reward_one'])[0], np.unique(behav_data['reward_two'])[0]
            signal = pd.DataFrame(ctb.dprime_metrics(behav_data, mouse, day=idx+1, reward_ports=[reward_one, reward_two], forward_reverse='forward'))
            signal['group'] = group
            signal['sex'] = sex
            metrics = pd.concat([metrics, signal], ignore_index=True)

In [ ]:
metrics.to_csv(pjoin(fig_path, 'intermediate_data/APPKI_signal_detection_metrics.csv'))

In [ ]:
## dprime across days
fig = pf.plot_behavior_across_days(data=metrics[metrics['day'] < 25], x_var='day', y_var='dprime', groupby_var=['day', 'group'], plot_transitions=[12.5],
                                   x_title='Day', y_title="d'", marker_color=two_color_plots, expert_line=False, chance=False, plot_datapoints=False)
fig.show()
fig.write_image(pjoin(fig_path, 'dprime_across_days.svg'))

In [ ]:
## During acquisition
anova_data = metrics.groupby(['mouse', 'group', 'day'], as_index=False).agg({'dprime': 'mean'})
adata = anova_data[anova_data['day'] < 13] ## start of reversal
adata.mixed_anova(dv='dprime', subject='mouse', within='day', between='group')

In [ ]:
## During reversal
anova_data = metrics.groupby(['mouse', 'group', 'day'], as_index=False).agg({'dprime': 'mean'})
adata = anova_data[(anova_data['day'] >= 13) & (anova_data['day'] < 25)] ## start of reversal
adata.mixed_anova(dv='dprime', subject='mouse', within='day', between='group')

In [ ]:
## Correct rejection across days
fig = pf.plot_behavior_across_days(data=metrics[metrics['day'] < 25], x_var='day', y_var='CR', groupby_var=['day', 'group'], plot_transitions=[12.5],
                                   x_title='Day', y_title="Correct Rejection Rate", marker_color=two_color_plots, expert_line=False, chance=False, plot_datapoints=False)
fig.add_hline(y=1, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.show()
fig.write_image(pjoin(fig_path, 'CR_across_days.svg'))

In [ ]:
## During acquisition
anova_data = metrics.groupby(['mouse', 'group', 'day'], as_index=False).agg({'CR': 'mean'})
adata = anova_data[anova_data['day'] < 13] ## start of reversal
adata.mixed_anova(dv='CR', subject='mouse', within='day', between='group')

In [ ]:
## During reversal
anova_data = metrics.groupby(['mouse', 'group', 'day'], as_index=False).agg({'CR': 'mean'})
adata = anova_data[(anova_data['day'] >= 13) & (anova_data['day'] < 25)] ## start of reversal
adata.mixed_anova(dv='CR', subject='mouse', within='day', between='group')

In [ ]:
## Correct rejection across days
fig = pf.plot_behavior_across_days(data=metrics[metrics['day'] < 25], x_var='day', y_var='hits', groupby_var=['day', 'group'], plot_transitions=[12.5],
                                   x_title='Day', y_title="Hit Rate", marker_color=two_color_plots, expert_line=False, chance=False, plot_datapoints=False)
fig.add_hline(y=1, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.show()
fig.write_image(pjoin(fig_path, 'HR_across_days.svg'))

In [ ]:
## During acquisition
anova_data = metrics.groupby(['mouse', 'group', 'day'], as_index=False).agg({'hits': 'mean'})
adata = anova_data[anova_data['day'] < 13] ## start of reversal
adata.mixed_anova(dv='hits', subject='mouse', within='day', between='group')

In [ ]:
## During reveresal
anova_data = metrics.groupby(['mouse', 'group', 'day'], as_index=False).agg({'hits': 'mean'})
adata = anova_data[(anova_data['day'] >= 13) & (anova_data['day'] < 25)] ## start of reversal
adata.mixed_anova(dv='hits', subject='mouse', within='day', between='group')

### Correct rejections and dprime across trials for a specified day.

In [ ]:
## Bin metrics
bin_size = 5
variable_of_interest = 'dprime'
binned_metrics = ctb.aggregate_metrics(metrics, bin_size=bin_size, variable_of_interest=variable_of_interest)
avg_binned = binned_metrics.groupby(['group', 'day', 'binned_trial'], as_index=False).agg({variable_of_interest: ['mean', 'sem']})
avg_binned

In [ ]:
## Plot variable of interest across trials
only_min_trials = True
day = 14
error_colors = ['rgba(255,0,0,0.4)', 'rgba(25,25,112,0.4)'] ## rgba is the only way to get the error band transparent
fig = pf.custom_graph_template(x_title='', y_title='', height=500, width=500,
                               titles=['Second Day Reversal'])
plot_data = avg_binned[avg_binned['day'] == day]

if only_min_trials:
    trial_lengths = []
    for group in np.unique(plot_data['group']):
        trial_lengths.append(plot_data['binned_trial'][plot_data['group'] == group].values[-1])
        min_trials = np.min(trial_lengths)

for idx, group in enumerate(np.unique(plot_data['group'])):
        if only_min_trials:
            group_data = plot_data[(plot_data['group'] == group) & (plot_data['binned_trial'] <= min_trials)]
            upper = group_data[variable_of_interest]['mean'] + group_data[variable_of_interest]['sem']
            lower = group_data[variable_of_interest]['mean'] - group_data[variable_of_interest]['sem']
        else:
            group_data = plot_data[(plot_data['group'] == group)]
            upper = group_data[variable_of_interest]['mean'] + group_data[variable_of_interest]['sem']
            lower = group_data[variable_of_interest]['mean'] - group_data[variable_of_interest]['sem']
        x_data = np.arange(1, len(group_data['binned_trial'])+1) * bin_size
        fig.add_trace(go.Scatter(x=x_data, y=group_data[variable_of_interest]['mean'], mode='lines', opacity=opacity,
                                line_color=group_colors_dict[group], showlegend=False,
                                legendgroup=group, name=group))
        fig.add_trace(go.Scatter(name='Upper Bound', x=x_data, y=upper, mode='lines',
                                 marker=dict(color=error_colors[idx]), line=dict(width=0), showlegend=False, 
                                 legendgroup=group))
        fig.add_trace(go.Scatter(name='Lower Bound', x=x_data, y=lower, marker=dict(color=error_colors[idx]),
                                 line=dict(width=0), mode='lines', fillcolor=error_colors[idx], fill='tonexty', 
                                 showlegend=False, legendgroup=group))
if variable_of_interest != 'dprime':
    fig.add_hline(y=1, line_width=1, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_yaxes(title="d'")
fig.update_xaxes(title='Trial')
fig.show()
fig.write_image(pjoin(fig_path, 'second_day_reversal_dprime_mintrial_wtapp.svg'))